In [6]:
import pandas as pd
import numpy as np

In [7]:
path = "/g/data/eg3/ab4502/TINTobjects/"
fids = ["8_20061216_20061216_aws.csv",
        "2_20100306_20100306_aws.csv",
        "50_20111007_20111007_aws.csv",
        "2_20150228_20150228_aws.csv",
        "71_20160114_20160114_aws.csv",
        "2_20161121_20161121_aws.csv",
        "2_20200131_20200131_aws.csv",
        "2_20200827_20200827_aws.csv"]

In [8]:
cols = ["time","scan","uid","area_km","vol","speed","speed_rnge","angle","conv_pct","azi_shear",
        "duration_mins","major_axis_length","eccentricity","field_max","local_max","max_alt"]

sums = {"scan":np.min,
        "time":np.min,
        "vol":np.median,
        "area_km":np.median,
        "field_max":np.max, 
        "min_alt":np.median,
        "max_alt":np.median,
        "local_max":np.median,
        "major_axis_length":np.median,
        "eccentricity":np.median,
        "angle":np.median,
        "speed":np.median,
        "speed_rnge":np.median,
        "duration_mins":np.median,
        "azi_shear":np.max,
        "conv_pct":np.median}

stats_inst = pd.DataFrame()
stats_all = pd.DataFrame()
stats_sum = pd.DataFrame()
gust = []
for f in fids:
    if f == "2_20200131_20200131_aws.csv":
        scan, uid = pd.read_csv(path+f).dropna(subset=["gust"]).sort_values("gust", ascending=False).query("in10km==1").iloc[1][["scan","uid"]].values
        gust.append(pd.read_csv(path+f).dropna(subset=["gust"]).sort_values("gust", ascending=False).query("in10km==1").iloc[1]["gust"])
    else:
        scan, uid = pd.read_csv(path+f).dropna(subset=["gust"]).sort_values("gust", ascending=False).query("in10km==1").iloc[0][["scan","uid"]].values
        gust.append(pd.read_csv(path+f).dropna(subset=["gust"]).sort_values("gust", ascending=False).query("in10km==1").iloc[0]["gust"])
    storm_df = pd.read_csv(path+f.replace("_aws",""))
    storm_df_agg = storm_df.groupby("uid").agg(sums)
    stats_inst = pd.concat([stats_inst, storm_df.query("(scan=="+str(scan)+") & (uid=="+str(uid)+")")[cols]], axis=0)
    stats_sum = pd.concat([stats_sum, storm_df_agg.loc[int(uid)]], axis=1)
    stats_all = pd.concat([stats_all, storm_df.query("(uid=="+str(uid)+")")[cols]], axis=0)  
stats_sum = stats_sum.T.reset_index().rename(columns={"index":"uid"}).set_index("time")
stats_inst = stats_inst.reset_index().drop(columns=["index"]).set_index("time")
stats_all = stats_all.set_index("uid")
stats_sum["gust"] = gust
stats_inst["gust"] = gust

In [9]:
for k in cols:
    if k not in ["uid","scan","time"]:
        stats_sum[k] = pd.to_numeric(stats_sum[k])
        stats_all[k] = pd.to_numeric(stats_all[k])
        stats_inst[k] = pd.to_numeric(stats_inst[k])

In [10]:
stats_sum[cols[1:]+["gust"]].round(1)

,scan,uid,area_km,vol,speed,speed_rnge,angle,conv_pct,azi_shear,duration_mins,major_axis_length,eccentricity,field_max,local_max,max_alt,gust
time,,,,,,,,,,,,,,,,
2006-12-16 04:21:40,26,63,9805.5,36306.0,16.8,0.8,39.1,0.5,NaN,520.2,215.1,0.9,73.2,16.0,12.2,54.5
2010-03-06 02:36:31,26,21,9042.0,26735.5,15.7,1.9,128.8,0.6,4.8,300.0,154.7,0.8,72.7,15.0,10.5,28.3
2011-10-07 17:01:46,100,130,21164.0,79911.5,15.9,5.4,81.1,0.6,NaN,410.0,222.7,0.8,68.5,30.0,11.5,35.0
2015-02-28 07:30:34,75,255,27200.0,103547.5,13.2,0.8,77.1,0.7,3.1,480.0,260.0,0.8,69.2,40.0,8.5,34.0
2016-01-14 03:37:00,18,72,1329.0,9360.0,19.4,19.9,136.1,0.8,5.3,54.0,90.6,1.0,66.4,3.0,14.5,33.4
2016-11-21 04:31:10,45,56,5238.0,14364.5,26.5,4.0,114.2,0.5,3.8,287.8,149.7,0.9,65.8,14.0,8.0,27.3
2020-01-31 01:48:28,18,19,183.0,672.8,23.3,0.6,151.6,0.6,2.2,210.0,20.7,0.8,59.9,1.0,6.0,33.1
2020-08-27 06:00:28,59,0,2671.5,2718.0,26.9,3.9,63.2,0.2,4.4,114.0,180.5,1.0,52.7,8.5,3.2,28.8


In [11]:
stats_inst[cols[1:]+["gust"]].round(1)

,scan,uid,area_km,vol,speed,speed_rnge,angle,conv_pct,azi_shear,duration_mins,major_axis_length,eccentricity,field_max,local_max,max_alt,gust
time,,,,,,,,,,,,,,,,
2006-12-16 08:01:48,45,63,16636.0,56381.5,16.8,0.8,39.1,0.5,NaN,520.2,261.4,0.9,65.0,24.0,13.5,54.5
2010-03-06 03:24:32,34,21,7487.0,22660.0,15.7,1.9,128.8,0.6,4.4,300.0,140.3,0.7,70.1,11.0,11.0,28.3
2011-10-07 19:21:46,114,130,11908.0,46381.0,15.9,5.4,81.1,0.6,NaN,410.0,207.1,0.9,65.3,20.0,11.0,35.0
2015-02-28 09:24:34,94,255,33772.0,139596.5,13.2,0.8,77.1,0.8,2.8,480.0,268.9,0.7,64.6,48.0,8.5,34.0
2016-01-14 04:19:00,25,72,1439.0,10360.0,19.4,19.9,136.1,0.8,3.2,54.0,95.4,1.0,65.2,4.0,15.0,33.4
2016-11-21 06:01:03,60,56,9950.0,32146.0,26.5,4.0,114.2,0.6,3.5,287.8,242.5,0.9,64.7,23.0,13.0,27.3
2020-01-31 03:42:28,37,19,366.0,1203.0,23.3,0.6,151.6,0.9,1.3,210.0,29.2,0.7,54.2,1.0,7.5,33.1
2020-08-27 07:12:28,71,0,3273.0,3166.5,26.9,3.9,63.2,0.1,2.8,114.0,179.1,1.0,48.2,9.0,2.5,28.8


In [18]:
data = stats_sum[cols[1:]+["gust"]].round(1)
data["class"] = "None"
data.loc[(data["major_axis_length"] >= 100) & (data["max_alt"]>=6), "class"] = "Linear"
data.loc[(data["major_axis_length"] >= 100) & (data["max_alt"]<6), "class"] = "Shallow linear"
data.loc[(data["major_axis_length"] < 100) & (data["max_alt"]>=6) & (data["azi_shear"]>=4), "class"] = "Supercell"
data.loc[(data["major_axis_length"] < 100) & (data["max_alt"]>=6) & (data["azi_shear"]<4), "class"] = "Cellular"
data.loc[(data["major_axis_length"] < 100) & (data["max_alt"]<6) & (data["azi_shear"]>=4), "class"] = "Shallow supercell"
data.loc[(data["major_axis_length"] < 100) & (data["max_alt"]<6) & (data["azi_shear"]<4), "class"] = "Shallow cellular"
data

,scan,uid,area_km,vol,speed,speed_rnge,angle,conv_pct,azi_shear,duration_mins,major_axis_length,eccentricity,field_max,local_max,max_alt,gust,class
time,,,,,,,,,,,,,,,,,
2006-12-16 04:21:40,26,63,9805.5,36306.0,16.8,0.8,39.1,0.5,NaN,520.2,215.1,0.9,73.2,16.0,12.2,54.5,Linear
2010-03-06 02:36:31,26,21,9042.0,26735.5,15.7,1.9,128.8,0.6,4.8,300.0,154.7,0.8,72.7,15.0,10.5,28.3,Linear
2011-10-07 17:01:46,100,130,21164.0,79911.5,15.9,5.4,81.1,0.6,NaN,410.0,222.7,0.8,68.5,30.0,11.5,35.0,Linear
2015-02-28 07:30:34,75,255,27200.0,103547.5,13.2,0.8,77.1,0.7,3.1,480.0,260.0,0.8,69.2,40.0,8.5,34.0,Linear
2016-01-14 03:37:00,18,72,1329.0,9360.0,19.4,19.9,136.1,0.8,5.3,54.0,90.6,1.0,66.4,3.0,14.5,33.4,Super-cellular
2016-11-21 04:31:10,45,56,5238.0,14364.5,26.5,4.0,114.2,0.5,3.8,287.8,149.7,0.9,65.8,14.0,8.0,27.3,Linear
2020-01-31 01:48:28,18,19,183.0,672.8,23.3,0.6,151.6,0.6,2.2,210.0,20.7,0.8,59.9,1.0,6.0,33.1,Cellular
2020-08-27 06:00:28,59,0,2671.5,2718.0,26.9,3.9,63.2,0.2,4.4,114.0,180.5,1.0,52.7,8.5,3.2,28.8,Shallow linear
